# Afternoon class 26/08 — Worksheet 07 SOLUTIONS: dictionaries   (L04)

Every cell below was executed on the same Python the lab ships; the quoted
output is real, including the two error messages.

Questions 3 and 9 raise on purpose, so running this notebook straight through
will stop at Q3. Run the cells after it individually.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 07 — Dictionaries. Run this once.
contact = {"name": "Ada", "phone": "416-555-0134", "city": "Toronto"}
grocery_items = {"milk": 2, "beef": 1, "eggs": 12, "bread": 3}
tweet = {
    "user": {"handle": "@dataninja", "followers": 1200},
    "text": "sets are just lists that refuse to repeat themselves",
    "tags": ["python", "data"],
}

print("contact:      ", contact)
print("grocery_items:", grocery_items)
print("tweet:        ", tweet)

PART A — Creating and reading a dictionary

### Question 1

Warm-up — three ways to build a dictionary. -> `{'brand': 'Ford', 'model': 'Mustang', 'year': 1964}` twice, then `{'brand': 0, 'model': 0, 'year': 0}`, then `<class 'dict'>`.

The literal and `dict()` produce an identical dictionary. Note the keys
are quoted strings in the first and bare keyword names in the second —
which is also `dict()`'s limitation: it only works when every key is a
valid Python name, so a key with a space, or `dict(2024=...)`, is
impossible.

`fromkeys` is for pre-seeding every key with the same starting value,
such as a set of counters at zero.

In [ ]:
car = {"brand": "Ford", "model": "Mustang", "year": 1964}
car2 = dict(brand="Ford", model="Mustang", year=1964)   # keys unquoted here
blank = dict.fromkeys(["brand", "model", "year"], 0)    # same keys, one value

print(car)
print(car2)
print(blank)
print(type(car))

### Question 2

Reading, including nested. -> `Ada`, `@dataninja`, `1200`, `python`.

`tweet['user']` returns another dictionary, so a second `['handle']`
reaches into it. `tweet['tags']` returns a list, so what comes next is a
NUMBER, not a key.

Real JSON is exactly this — dictionaries and lists nested arbitrarily —
and you follow the shape one bracket at a time. When you lose your place,
print the intermediate step rather than guessing at the full path.

In [ ]:
print(contact["name"])

print(tweet["user"]["handle"])      # dict inside a dict -- key, then key
print(tweet["user"]["followers"])
print(tweet["tags"][0])             # list inside a dict -- key, then index

### Question 3

The missing key. -> `KeyError: 'email'`.

Not a `None`, not an empty string, not a warning — a hard stop. Python
refuses to guess what you meant by a key that is not there.

That is a feature far more often than an annoyance: it fails on the line
with the actual mistake, rather than letting a `None` travel through three
more steps before something unrelated breaks in a way that no longer
points at the cause.

In [ ]:
# This is SUPPOSED to raise. Asking for a key that is not there is an error,
# not a None and not a blank.
print(contact["email"])

### Question 4

`.get()`, the safe version. -> `None`, `none on file`, `2`, `-1`.

`.get()` never raises. With no default it returns `None`; with one it
returns whatever you nominated. The `milk` lookup shows the default is
ignored when the key does exist.

WHICH TO USE: `[]` when a missing key means something is genuinely wrong
and you want to hear about it immediately. `.get()` when absence is a
normal, expected case and you have a sensible fallback. Wrapping
everything in `.get()` to avoid ever seeing an error does not remove the
problem, it just relocates the failure somewhere less informative.

In [ ]:
print(contact.get("email"))                    # missing -> None, no error
print(contact.get("email", "none on file"))    # missing -> your default

print(grocery_items.get("milk", -1))     # present, so the default is ignored
print(grocery_items.get("caviar", -1))   # absent, so you get -1

# Use .get() when a missing key is normal and you have a sensible fallback.
# Use [] when a missing key means something is genuinely wrong and you would
# rather find out loudly than carry on with a None.

PART B — Changing a dictionary

### Question 5

Adding, changing, deleting. -> email is added; city becomes `Ottawa`; phone disappears, leaving `{'name': 'Ada', 'city': 'Ottawa', 'email': 'ada@example.com'}`.

`contact['email'] = ...` and `contact['city'] = ...` are the same
statement. Whether it adds or overwrites depends only on whether the key
already existed, and Python will not tell you which happened.

The consequence is worth internalising: a typo in a key name silently
creates a new entry instead of updating the one you meant, and the
dictionary looks fine afterwards.

In [ ]:
contact["email"] = "ada@example.com"   # key is new -> adds it
print(contact)

contact["city"] = "Ottawa"             # key exists -> replaces the value
print(contact)

del contact["phone"]
print(contact)

### Question 6

update, pop, popitem, clear. -> `update` gives 5 pairs, with `milk` now 3 and `rice` added; `pop('eggs')` returns `12`; `popitem()` returns `('rice', 1)`; the copy prints `{}` while the original keeps `{'milk': 3, 'beef': 1, 'bread': 3}`.

One `update` call both changed `milk` and added `rice` — the same
add-or-overwrite rule as Q5, applied in bulk and just as silent about
which it did to each key.

`pop(key)` returns the VALUE. `popitem()` takes no argument and returns
the whole `(key, value)` pair, removing the last-inserted one. Dictionaries
have kept insertion order since Python 3.7, so "last" is well defined —
but do not lean on that ordering to sort anything.

In [ ]:
grocery_items.update({"milk": 3, "rice": 1})   # milk exists, rice does not
print(grocery_items)

eggs = grocery_items.pop("eggs")   # pop RETURNS the value it removed
print("popped eggs:", eggs)
print(grocery_items)

last = grocery_items.popitem()     # removes and returns the LAST pair
print("popitem:", last)
print(grocery_items)

spare = grocery_items.copy()
spare.clear()
print(spare, grocery_items)

PART C — Views, keys, and iteration

### Question 7

Views. -> `dict_keys([...])`, `dict_values([...])`, `dict_items([...])`, then `3`, then `['name', 'city', 'email']` twice.

The first three do not print as lists because they are not lists. They are
VIEWS: live windows onto the dictionary that update themselves when it
changes, and that you cannot index. `list(...)` takes a snapshot.

Note the last two lines are identical. `list(a_dict)` gives you the KEYS,
because iterating a dictionary iterates its keys — so `list(d)` and
`list(d.keys())` are the same thing, and the shorter one is idiomatic.

In [ ]:
print(contact.keys())     # a VIEW object, not a list
print(contact.values())
print(contact.items())    # pairs, as tuples
print(len(contact))       # counts the pairs

print(list(contact.keys()))   # wrap it to get a real list
print(list(contact))          # list(dict) gives the KEYS -- a useful shortcut

# The views print as dict_keys([...]) because they are live windows onto the
# dictionary, not snapshots: change the dict and the view changes with it.
# You cannot index them. Wrap in list() when you need a real list.

### Question 8

Two ways to loop. -> `milk`, `beef`, `bread`; then `milk: 3`, `beef: 1`, `bread: 3`.

Looping a dictionary directly gives KEYS only, which surprises people who
expected values. `.items()` unpacks into two names right in the `for`
statement — the same tuple unpacking you used in worksheet 03 Q7, and it
works for the same reason: each item genuinely IS a `(key, value)` tuple.

If you catch yourself writing `for k in d: v = d[k]`, `.items()` is the
line you actually wanted.

In [ ]:
for key in grocery_items:      # iterating a dict yields its KEYS
    print(key)

print("---")

for item, quantity in grocery_items.items():   # unpacking each (k, v) tuple
    print(f"{item}: {quantity}")

### Question 9

Three rules about keys. -> `{'a': 99, 'b': 2}`; then `{(43.65, -79.38): 'Toronto'}`; then `TypeError: unhashable type: 'list'`.

DUPLICATE KEYS: no error and no warning. The second `'a'` overwrote the
first and the dictionary reports two pairs, not three. Keys are unique by
definition, so "last one wins" is the only possible outcome — but when
that literal comes from generated code or a merge, you have silently lost
data and the result looks perfectly healthy.

WHY A TUPLE WORKS AND A LIST DOES NOT: a dictionary locates a key by
hashing it, which requires the key never to change. A tuple cannot change,
so it can be hashed; a list can, so it cannot. This is the practical payoff
of worksheet 03 — immutability is not a restriction here, it is the ticket
that lets tuples serve as keys at all.

In [ ]:
print({"a": 1, "b": 2, "a": 99})
# Not an error and not a warning -- the later "a" simply overwrote the
# earlier one. Keys are unique by definition, so the last one wins.

by_location = {(43.65, -79.38): "Toronto"}   # a TUPLE key is fine
print(by_location)

# This is SUPPOSED to raise. A key must be immutable, and a list is not.
by_location[[43.65, -79.38]] = "Toronto"

### Question 10

`in` searches keys. -> `True`, `False`, `True`.

`3` really is in the dictionary — it is `milk`'s quantity — and yet
`3 in grocery_items` is `False`, because `in` on a dict asks about KEYS.

That is not an inconsistency, it is the entire point of the structure. A
dictionary is built to answer "do you have this key" instantly. Answering
"does this value appear anywhere" means scanning every pair, which is why
you have to ask for it explicitly with `.values()`.

In [ ]:
print("milk" in grocery_items)   # "milk" is a KEY
print(3 in grocery_items)        # 3 is a VALUE, so this is False

print(3 in grocery_items.values())   # ask the values explicitly

# `in` on a dict tests keys, because that is the lookup a dict is built for.
# Searching the values means scanning every pair, which is why you have to
# ask for it by name.

### Question 11

Stretch — dict comprehensions, and the inversion trap. -> `{'milk': 6, 'beef': 2, 'bread': 6}`; `{'milk': 3, 'bread': 3}`; `{3: 'bread', 1: 'beef'}`; and the two lengths are `3` and `2`.

LOOK AT THE LAST LINE. Inverting a 3-pair dictionary produced a 2-pair
one. `milk` and `bread` both had quantity 3, so both became the key `3`,
and the later one silently won — exactly the duplicate-key rule from Q9,
turning up where you were not looking for it.

The inverted dictionary is not corrupt and nothing failed. It is a
correct answer to a question you probably did not mean to ask. Inverting is
only safe when the values are known to be unique, and "known" has to mean
checked: compare `len()` before and after, every time.

In [ ]:
doubled = {k: v * 2 for k, v in grocery_items.items()}
print(doubled)

plentiful = {k: v for k, v in grocery_items.items() if v > 1}
print(plentiful)

by_quantity = {v: k for k, v in grocery_items.items()}   # keys <-> values
print(by_quantity)

print(len(grocery_items), len(by_quantity))